In [75]:
import pandas as pd 
import numpy as np

In [76]:
movies=pd.read_csv('C:\\Users\\mayan\\Videos\\Movie-Recommendation\\Dataset\\tmdb_5000_movies.csv')

In [77]:
credits=pd.read_csv('C:\\Users\\mayan\\Videos\\Movie-Recommendation\\Dataset\\tmdb_5000_credits.csv')

In [78]:
movies.shape

(4803, 20)

In [79]:
credits.shape

(4803, 4)

In [80]:
movies=movies.merge(credits,on='title')

In [81]:
movies.shape

(4809, 23)

In [82]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4809 entries, 0 to 4808
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4809 non-null   int64  
 1   genres                4809 non-null   object 
 2   homepage              1713 non-null   object 
 3   id                    4809 non-null   int64  
 4   keywords              4809 non-null   object 
 5   original_language     4809 non-null   object 
 6   original_title        4809 non-null   object 
 7   overview              4806 non-null   object 
 8   popularity            4809 non-null   float64
 9   production_companies  4809 non-null   object 
 10  production_countries  4809 non-null   object 
 11  release_date          4808 non-null   object 
 12  revenue               4809 non-null   int64  
 13  runtime               4807 non-null   float64
 14  spoken_languages      4809 non-null   object 
 15  status               

In [83]:
movies=movies[['movie_id','title','overview','genres','keywords','cast','crew']]

In [84]:
movies.isnull().sum()

movie_id    0
title       0
overview    3
genres      0
keywords    0
cast        0
crew        0
dtype: int64

In [85]:
movies.dropna(inplace=True)

In [86]:
movies.shape

(4806, 7)

In [87]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4806 entries, 0 to 4808
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   movie_id  4806 non-null   int64 
 1   title     4806 non-null   object
 2   overview  4806 non-null   object
 3   genres    4806 non-null   object
 4   keywords  4806 non-null   object
 5   cast      4806 non-null   object
 6   crew      4806 non-null   object
dtypes: int64(1), object(6)
memory usage: 300.4+ KB


In [88]:
movies.duplicated().sum()

0

In [89]:
movies.iloc[0].genres

'[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]'

In [90]:
import ast
ast.literal_eval(movies.iloc[0].genres)

[{'id': 28, 'name': 'Action'},
 {'id': 12, 'name': 'Adventure'},
 {'id': 14, 'name': 'Fantasy'},
 {'id': 878, 'name': 'Science Fiction'}]

In [91]:

def conversion(obj):
    l=[]
    for i in ast.literal_eval(obj):
        l.append(i['name'])
    return l
        
    

In [92]:
movies['keywords']=movies['keywords'].apply(conversion)

In [93]:
movies['genres']=movies['genres'].apply(conversion)

In [94]:
def castconversion(obj):
    l=[]
    counter=0
    for i in ast.literal_eval(obj):
        if counter<5:
          l.append(i['name'])
        counter=counter+1
    return l

In [95]:
movies['cast']=movies['cast'].apply(castconversion)

In [96]:
movies.head(1)

,movie_id,title,overview,genres,keywords,cast,crew
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is di...","[Action, Adventure, Fantasy, Science Fiction]","[culture clash, future, space war, space colon...","[Sam Worthington, Zoe Saldana, Sigourney Weave...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."


In [97]:
def crewconversion(obj):
    l=[]
    for i in ast.literal_eval(obj):
        if(i['job']=='Director'):
          l.append(i['name'])
    return l

In [98]:
movies['crew']=movies['crew'].apply(crewconversion)

In [99]:
def collapse(obj):
    l=[]
    for i in obj:
        l.append(i.replace(" ",""))
    return l

In [100]:
movies['cast']=movies['cast'].apply(collapse)
movies['genres']=movies['genres'].apply(collapse)
movies['crew']=movies['crew'].apply(collapse)
movies['keywords']=movies['keywords'].apply(collapse)


In [101]:
movies['overview']=movies['overview'].apply(lambda x:x.split())

In [102]:
movies['tags']=movies['overview']+movies['genres']+movies['keywords']+movies['cast']+movies['crew']

In [104]:
new_movies=movies.drop(columns=['overview','genres','keywords','cast','crew'])

In [106]:
new_movies['tags']=new_movies['tags'].apply(lambda x:" ".join(x))

In [107]:
new_movies['tags']=new_movies['tags'].apply(lambda x:x.lower())

In [109]:
import nltk
from nltk.stem.porter import PorterStemmer

In [110]:
ps=PorterStemmer()
def stem(text):
    y=[]
    for i in text.split():
        y.append(ps.stem(i))
    return " ".join(y)
        
    

In [111]:
new_movies['tags']=new_movies['tags'].apply(stem)

In [112]:
from sklearn.feature_extraction.text import CountVectorizer

In [114]:
cv=CountVectorizer(max_features=5000,stop_words='english')

In [116]:
vector=cv.fit_transform(new_movies['tags']).toarray()

In [117]:
from sklearn.metrics.pairwise import cosine_similarity

In [118]:
similarity=cosine_similarity(vector)

In [125]:
def recommend(Movie):
     index= new_movies[new_movies['title']==Movie].index[0]
     l=sorted(list(enumerate(similarity[index])),reverse=True,key=lambda x:x[1])
     for i in l[1:6]:
         print(new_movies.iloc[i[0]].title)

In [126]:
recommend("Avatar")

Aliens vs Predator: Requiem
Independence Day
Falcon Rising
Battle: Los Angeles
Titan A.E.


In [133]:
import pickle
pickle.dump(new_movies, open("Model/movies_list.pkl", "wb"))
